# Домашнее задание 3. Статистическая проверка качества регрессии

Данные разделены на Train / Validation / Test по дополнительному указанию преподавателя. Модель обучается только на Train; Validation используется для анализа и проверки удаления признаков; Test — один раз в конце.


### 1. Загрузите файл candidates.csv с компьютера. 

* откройте первые несколько строк
* изучите строение файла.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# В файле разделитель «;», а десятичный разделитель — запятая.
df = pd.read_csv("candidates.csv", sep=";", decimal=",", encoding="utf-8-sig")

display(df.head())
print("Размер таблицы:", df.shape)
print("\nТипы столбцов:")
print(df.dtypes)


,Код претендента,Рейтинг университета окончания,Средний балл диплома,Опыт работы (лет),Возраст,Рейтинг университета диплома,Баллы за сертификаты,Желаемая заработная плата,Итоговая зарплата от компании
0,1001,82,3.22,10,33,84,89,237000,237000
1,1002,4,3.19,3,29,16,0,155000,155000
2,1003,70,3.84,6,30,70,74,212000,197000
3,1004,98,3.67,2,24,97,64,140000,130000
4,1005,94,3.92,2,27,99,66,146000,136000


Размер таблицы: (800, 9)

Типы столбцов:
Код претендента                     int64
Рейтинг университета окончания      int64
Средний балл диплома              float64
Опыт работы (лет)                   int64
Возраст                             int64
Рейтинг университета диплома        int64
Баллы за сертификаты                int64
Желаемая заработная плата           int64
Итоговая зарплата от компании       int64
dtype: object


### 2. На основе файла создайте матрицу X с характеристиками (некоторые столбцы могут не понадобятся для характеристик, подумайте какие), и столбец y (из последнего столбца).

* отобразите первые строки матрицы X и столбца y на экране.

In [2]:
# «Код претендента» — идентификатор; последний столбец — целевая переменная.
target_name = df.columns[-1]
feature_names = list(df.columns[1:-1])
X = df[feature_names].to_numpy(dtype=float)
y = df[target_name].to_numpy(dtype=float)

print("Первые строки X:")
display(pd.DataFrame(X, columns=feature_names).head())
print("Первые значения y:")
display(pd.Series(y, name=target_name).head())

# 60% Train, 20% Validation, 20% Test. Test не используется до итоговой проверки.
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=42
)
print(f"Размеры: Train={len(y_train)}, Validation={len(y_val)}, Test={len(y_test)}")


Первые строки X:


,Рейтинг университета окончания,Средний балл диплома,Опыт работы (лет),Возраст,Рейтинг университета диплома,Баллы за сертификаты,Желаемая заработная плата
0,82.0,3.22,10.0,33.0,84.0,89.0,237000.0
1,4.0,3.19,3.0,29.0,16.0,0.0,155000.0
2,70.0,3.84,6.0,30.0,70.0,74.0,212000.0
3,98.0,3.67,2.0,24.0,97.0,64.0,140000.0
4,94.0,3.92,2.0,27.0,99.0,66.0,146000.0


Первые значения y:


0    237000.0
1    155000.0
2    197000.0
3    130000.0
4    136000.0
Name: Итоговая зарплата от компании, dtype: float64

Размеры: Train=480, Validation=160, Test=160


### 3. Используя SAG, постройте на этих данных модель линейной регрессии

* выведите на экран полученное уравнение линиии

In [3]:
def make_sag_model():
    # Масштабирование обучается внутри Pipeline только на переданном train.
    # alpha=0 отключает регуляризацию; solver=sag использует алгоритм SAG.
    return make_pipeline(
        StandardScaler(),
        Ridge(alpha=0.0, solver="sag", max_iter=100000, tol=1e-10, random_state=42),
    )


def original_scale_params(model):
    """Возвращает b0 и коэффициенты в исходных единицах."""
    scaler = model.named_steps["standardscaler"]
    reg = model.named_steps["ridge"]
    coef = reg.coef_ / scaler.scale_
    intercept = reg.intercept_ - np.dot(coef, scaler.mean_)
    return float(intercept), np.asarray(coef, dtype=float)


def print_equation(intercept, coef, names):
    equation = f"ŷ = {intercept:.6f}"
    for c, name in zip(coef, names):
        sign = "+" if c >= 0 else "-"
        equation += f" {sign} {abs(c):.6f}·[{name}]"
    print(equation)


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def r2_by_formula(y_true, y_pred):
    return 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2)


model = make_sag_model()
model.fit(X_train, y_train)
intercept, coef = original_scale_params(model)

print("Уравнение исходной модели (обучение только на Train):")
print_equation(intercept, coef, feature_names)
val_rmse_before = rmse(y_val, model.predict(X_val))
print(f"RMSE на Validation до отбора признаков: {val_rmse_before:.3f} руб.")


Уравнение исходной модели (обучение только на Train):
ŷ = 32596.857061 - 181.753257·[Рейтинг университета окончания] + 11519.660085·[Средний балл диплома] + 9391.090013·[Опыт работы (лет)] + 171.351494·[Возраст] - 166.789590·[Рейтинг университета диплома] + 238.997878·[Баллы за сертификаты] + 0.335557·[Желаемая заработная плата]
RMSE на Validation до отбора признаков: 8080.240 руб.


### 4. Проверьте веса на значимость, используя алгоритм и формулы из лекции

* код должен соотвествовать алгоритму из лекции
* объясните логически, почему мы проверяем две гипотезы
* какие веса у вас оказались незначимыми?

In [4]:
# t-проверка коэффициентов обученной модели: по train, иначе t-распределение
# для обученных на train коэффициентов применять к val-остаткам некорректно.
alpha = 0.05
n, k = X_train.shape
X_design = np.column_stack([np.ones(n), X_train])
y_pred_train = model.predict(X_train)

# s_e^2 = SSE_train/(n-k-1), C = s_e^2 * (X_design^T X_design)^(-1).
sse_train = np.sum((y_train - y_pred_train) ** 2)
s_e2 = sse_train / (n - k - 1)
C = s_e2 * np.linalg.inv(X_design.T @ X_design)
standard_errors = np.sqrt(np.diag(C))

all_coef = np.concatenate([[intercept], coef])
t_values = all_coef / standard_errors

t_left = stats.t.ppf(alpha / 2, df=n - k - 1)
t_right = stats.t.ppf(1 - alpha / 2, df=n - k - 1)

rows = []
coef_names = ["Свободный член b0"] + feature_names
for name, b, t_value in zip(coef_names, all_coef, t_values):
    significant = not (t_left <= t_value <= t_right)
    rows.append([name, b, t_value, "значим" if significant else "НЕЗНАЧИМ"])

significance_table = pd.DataFrame(
    rows, columns=["Коэффициент", "Значение", "t-статистика", "Вывод"]
)
display(significance_table)
print(f"Критический t-интервал, df={n-k-1}: [{t_left:.6f}; {t_right:.6f}]")

# Отбор только среди характеристик, без свободного члена.
insignificant_features = [
    name for name, t_value in zip(feature_names, t_values[1:])
    if t_left <= t_value <= t_right
]
print("Незначимые характеристики:", insignificant_features)


,Коэффициент,Значение,t-статистика,Вывод
0,Свободный член b0,32596.857061,4.196599,значим
1,Рейтинг университета окончания,-181.753257,-3.222015,значим
2,Средний балл диплома,11519.660085,15.069394,значим
3,Опыт работы (лет),9391.090013,23.444927,значим
4,Возраст,171.351494,0.594959,НЕЗНАЧИМ
5,Рейтинг университета диплома,-166.789590,-3.025744,значим
6,Баллы за сертификаты,238.997878,16.683004,значим
7,Желаемая заработная плата,0.335557,20.463064,значим


Критический t-интервал, df=472: [-1.965003; 1.965003]
Незначимые характеристики: ['Возраст']


**Пояснение к п. 4.** Проверяем $H_0: b_i=0$ против $H_1: b_i\ne0$. t-статистики вычисляются из коэффициентов и остатков на **Train** с $\alpha=0{,}05$ и $df=n-k-1$. Наличие дополнительной Validation не отменяет необходимости оценивать стандартные ошибки коэффициентов по обучающей выборке.


### 5. Если какие-то веса оказались незначимыми, перепишите матрицу характеристик X без соответствующих столбцов, обучите заново модель, используя SAG

In [5]:
# Удаляем незначимые признаки и заново обучаем SAG только на Train.
old_feature_names = feature_names.copy()
feature_names = [name for name in old_feature_names if name not in insignificant_features]
keep_idx = [old_feature_names.index(name) for name in feature_names]
X_train = X_train[:, keep_idx]
X_val = X_val[:, keep_idx]

model = make_sag_model()
model.fit(X_train, y_train)
intercept, coef = original_scale_params(model)

print("Характеристики после удаления незначимых:")
for name in feature_names:
    print("-", name)
print("\nНовое уравнение модели:")
print_equation(intercept, coef, feature_names)

# Оцениваем изменение на Validation, НЕ на Test.
val_rmse_after = rmse(y_val, model.predict(X_val))
print(f"\nRMSE Validation до удаления: {val_rmse_before:.3f} руб.")
print(f"RMSE Validation после удаления: {val_rmse_after:.3f} руб.")
print(f"Разница RMSE (после - до): {val_rmse_after-val_rmse_before:+.3f} руб.")
print("Вывод: сравниваем качество на Validation; удаление выполнялось по t-критерию из задания.")


Характеристики после удаления незначимых:
- Рейтинг университета окончания
- Средний балл диплома
- Опыт работы (лет)
- Рейтинг университета диплома
- Баллы за сертификаты
- Желаемая заработная плата

Новое уравнение модели:
ŷ = 36772.582864 - 179.762620·[Рейтинг университета окончания] + 11519.122224·[Средний балл диплома] + 9561.818727·[Опыт работы (лет)] - 168.595870·[Рейтинг университета диплома] + 238.787923·[Баллы за сертификаты] + 0.335387·[Желаемая заработная плата]

RMSE Validation до удаления: 8080.240 руб.
RMSE Validation после удаления: 8088.050 руб.
Разница RMSE (после - до): +7.810 руб.
Вывод: сравниваем качество на Validation; удаление выполнялось по t-критерию из задания.


### 6. Проверьте модель на значимость по алгоритму из лекции

* код должен соответствовать алгоритму из лекции
* объясните логически, почему мы проверяем две гипотезы и что эти гипотезы собой подразумевают
* какая у вас получилась модель?
* подходит ли она для описания наших данных?

In [6]:
# F-проверка статистической значимости проводится на Train, на котором оценены
# коэффициенты; R² на Validation оценивает предсказательную способность отдельно.
n, k = X_train.shape
y_pred_train = model.predict(X_train)
R2_train = r2_by_formula(y_train, y_pred_train)

F = (R2_train / k) / ((1 - R2_train) / (n - k - 1))
F_critical = stats.f.ppf(1 - alpha, dfn=k, dfd=n - k - 1)
model_significant = F > F_critical

print(f"R² Train = {R2_train:.6f}")
print(f"F = {F:.6f}")
print(f"F_critical(1-alpha={1-alpha:.2f}; {k}; {n-k-1}) = {F_critical:.6f}")
print("Вывод:", "H0 отвергается — модель значима." if model_significant
      else "Нет оснований отвергнуть H0 — модель незначима.")
print(f"RMSE Validation = {rmse(y_val, model.predict(X_val)):.3f} руб.")


R² Train = 0.985617
F = 5402.061288
F_critical(1-alpha=0.95; 6; 473) = 2.117739
Вывод: H0 отвергается — модель значима.
RMSE Validation = 8088.050 руб.


**Пояснение к п. 6.** $H_0: b_1=\dots=b_k=0$ — все коэффициенты при признаках нулевые; $H_1$ — хотя бы один ненулевой. По F-критерию проверяем значимость коэффициентов на **Train** (с $df_1=k$, $df_2=n-k-1$); качество переносимости модели отдельно анализируется на **Validation**. Нельзя без обоснования применять ту же F-формулу с распределением Фишера к $R^2$ на Validation для модели, обученной на Train.


### 7. Рассчитайте парный коэффициент корреляции для каждой пары x_i, y, используя алгоритм и формулы из лекции.

* код должен соотвествовать алгоритму из лекции
* ответьте на вопрос "что в общем случае (а не в конкретно нашем) показывают парные коэффициенты корреляции?"
* выведите коэффициент корреляции на экран для каждой пары в виде: пара, значение коэффициента
* ответьте на вопрос "что полученные значения говорят о нашей модели?"
* какие характеристики оказывают на ответ наибольшее значение, какие наименьшее?

In [7]:
# Анализ парных линейных связей на Validation (не используем Test).
def pair_correlation(x, y):
    x_mean = np.mean(x)
    y_mean = np.mean(y)
    sigma_x = np.sqrt(np.mean((x - x_mean) ** 2))
    sigma_y = np.sqrt(np.mean((y - y_mean) ** 2))
    return np.mean((x - x_mean) * (y - y_mean)) / (sigma_x * sigma_y)

correlations = {}
for i, name in enumerate(feature_names):
    r = pair_correlation(X_val[:, i], y_val)
    correlations[name] = r
    print(f"{name} — {target_name}: r = {r:.6f}")

max_feature = max(correlations, key=lambda name: abs(correlations[name]))
min_feature = min(correlations, key=lambda name: abs(correlations[name]))
print(f"\nНаибольшая связь по |r| на Validation: {max_feature}, r = {correlations[max_feature]:.6f}")
print(f"Наименьшая связь по |r| на Validation: {min_feature}, r = {correlations[min_feature]:.6f}")


Рейтинг университета окончания — Итоговая зарплата от компании: r = -0.274253
Средний балл диплома — Итоговая зарплата от компании: r = 0.160767
Опыт работы (лет) — Итоговая зарплата от компании: r = 0.933289
Рейтинг университета диплома — Итоговая зарплата от компании: r = -0.252212
Баллы за сертификаты — Итоговая зарплата от компании: r = 0.190067
Желаемая заработная плата — Итоговая зарплата от компании: r = 0.968372

Наибольшая связь по |r| на Validation: Желаемая заработная плата, r = 0.968372
Наименьшая связь по |r| на Validation: Средний балл диплома, r = 0.160767


**Пояснение к п. 7.** Парная корреляция показывает силу и направление **линейной связи** между одним признаком и зарплатой. Корреляции рассчитаны по Validation, поэтому их численные значения могут отличаться от результатов для всех 800 объектов. Парная корреляция не эквивалентна t-значимости коэффициента множественной регрессии.


### 8. Вычислите коэффициент детерминации

* код должен соответствовать формуле из лекции
* выведите полученный коэффициент на экран
* что он говорит о нашей модели? насколько качественно она описала данные? лучше или хуже она, чем средняя линия?

In [8]:
# Коэффициент детерминации по формуле из лекции.
# Для анализа прогноза выводим прежде всего R² на Validation.
R2_train = r2_by_formula(y_train, model.predict(X_train))
R2_val = r2_by_formula(y_val, model.predict(X_val))

print(f"R² Train = {R2_train:.6f}")
print(f"R² Validation = {R2_val:.6f}")
print("Validation:", "лучше предсказания среднего на Validation" if R2_val > 0
      else "не лучше предсказания среднего на Validation")


R² Train = 0.985617
R² Validation = 0.985402
Validation: лучше предсказания среднего на Validation


**Пояснение к п. 8.** $R^2=1-\frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}$ сравнивает ошибки модели с предсказанием среднего. На Validation оценивается качество на объектах, не использованных при обучении. $R^2$ — **не процент точности предсказаний**.


### 9. Рассчитайте LOOCV кросс-валидации, используя алгоритм и формулы из лекции.

* код должен соотвествовать алгоритму из лекции
* объясните устно как он работает и зачем он нужен
* выведите LOOCV на экран
* ответьте на вопрос "что полученный LOOCV говорит о нашей модели? она идеально нам подходит, или в ней нужно что-то менять?"

In [9]:
# LOOCV из лекции выполняем ВНУТРИ Train, не включая Validation и Test.
errors = np.empty(len(y_train), dtype=float)

for i in range(len(y_train)):
    train_mask = np.ones(len(y_train), dtype=bool)
    train_mask[i] = False
    loocv_model = make_sag_model()
    loocv_model.fit(X_train[train_mask], y_train[train_mask])
    y_i_pred = loocv_model.predict(X_train[i:i + 1])[0]
    errors[i] = y_i_pred - y_train[i]

LOOCV = np.sqrt(np.mean(errors ** 2))
train_rmse = rmse(y_train, model.predict(X_train))

print(f"LOOCV (только Train) = {LOOCV:.3f} руб.")
print(f"RMSE Train = {train_rmse:.3f} руб.")
print(f"RMSE Validation = {val_rmse_after:.3f} руб.")
print(f"LOOCV составляет {LOOCV / np.mean(y_train) * 100:.2f}% средней зарплаты на Train.")


LOOCV (только Train) = 8972.500 руб.
RMSE Train = 8832.410 руб.
RMSE Validation = 8088.050 руб.
LOOCV составляет 3.84% средней зарплаты на Train.


**Пояснение к п. 9.** LOOCV по очереди оставляет один объект из Train для проверки и обучает модель на остальных $n_{train}-1$. Итог — корень из среднего квадрата ошибок на оставленных объектах. Так мы оцениваем устойчивость модели **внутри Train**; внешние Validation и Test не используются. Значение LOOCV сопоставляем с ошибкой на Validation. Test применяется только в заключительном пункте.


### Итоговая проверка на Test (дополнительное требование преподавателя)

После анализа и корректировки на Validation один раз оцениваем фиксированную модель на ранее неиспользованной Test-выборке. Не подбираем параметры и не удаляем признаки по результату Test.

In [10]:
# К этому моменту состав признаков и параметры обучения уже зафиксированы.
# Test не участвовал ни в обучении, ни в подборе структуры модели.
X_test_final = X_test[:, keep_idx]
y_test_pred = model.predict(X_test_final)
test_rmse = rmse(y_test, y_test_pred)
test_r2 = r2_by_formula(y_test, y_test_pred)

print(f"Финальная ошибка RMSE Test = {test_rmse:.3f} руб.")
print(f"R² Test = {test_r2:.6f}")

Финальная ошибка RMSE Test = 8741.657 руб.
R² Test = 0.985911
